In [1]:
!pip install requests
!pip install dotenv

In [2]:
import requests
import json
import time
import re  # Regex for extracting Planner Plan ID from URLs
import urllib.parse
from dotenv import load_dotenv
import os



In [3]:
# Load environment variables
load_dotenv()
# Access token from .env
ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")


headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Accept": "application/json"
}

In [4]:
# Store results
channel_planners_detected = []

# Microsoft Graph API base URL
graph_base_url = "https://graph.microsoft.com/v1.0"

# Function to extract the real Plan ID from a Planner URL
def extract_plan_id_from_planner_url(planner_url: str) -> str:
    if not planner_url:
        return ""
    
    decoded_url = urllib.parse.unquote(planner_url)
    parsed = urllib.parse.urlparse(decoded_url)
    qs = urllib.parse.parse_qs(parsed.query)
    
    real_planner_link = qs["webUrl"][0] if "webUrl" in qs and qs["webUrl"] else decoded_url
    
    plan_id_match = re.search(r'/PlanViews/([^?&]+)', real_planner_link)
    if plan_id_match:
        return plan_id_match.group(1)
    
    plan_id_match = re.search(r'[?&]planId=([^&]+)', real_planner_link)
    if plan_id_match:
        return plan_id_match.group(1)
    
    return ""

# Function to clean "tt." formatted Planner IDs
def clean_plan_id(plan_id):
    if not plan_id:
        return None
    match = re.search(r"tt\.([a-zA-Z0-9-_]+)", plan_id)
    return f"tt.{match.group(1)}" if match else None

# Function to check if a tab is a valid Planner Tab
def is_planner_tab(tab_name, planner_id, planner_url):
    if planner_id and planner_id.startswith("tt."):
        return True
    if "planner" in tab_name.lower() or "tasks" in tab_name.lower():
        return True
    if planner_url and "tasks.office.com" in planner_url:
        return True
    return False

# Function to get all teams
def get_all_teams(headers):
    teams_url = f"{graph_base_url}/me/joinedTeams"
    response = requests.get(teams_url, headers=headers)
    if response.status_code == 200:
        return response.json().get("value", [])
    return []

# Function to get all channels for a given team
def get_team_channels(team_id, headers):
    channels_url = f"{graph_base_url}/teams/{team_id}/channels"
    response = requests.get(channels_url, headers=headers)
    if response.status_code == 200:
        return response.json().get("value", [])
    return []

# Function to get all tabs for a given channel
def get_channel_tabs(team_id, channel_id, headers):
    tabs_url = f"{graph_base_url}/teams/{team_id}/channels/{channel_id}/tabs"
    response = requests.get(tabs_url, headers=headers)
    if response.status_code == 200:
        return response.json().get("value", [])
    return []

# Function to split planner_tabs into separate objects
def split_planner_tabs(data):
    separated_items = []
    
    base_item = {
        "project_name": data["project_name"],
        "team_id": data["team_id"],
        "channel_name": data["channel_name"],
        "channel_id": data["channel_id"],
    }
    
    for tab in data["planner_tabs"]:
        new_item = base_item.copy()
        new_item["planner_tabs"] = [tab]  # Ensure it's a list with one element
        separated_items.append(new_item)
    
    return separated_items

In [5]:
# Fetch all teams
teams = get_all_teams(headers)

#print(teams)

for team in teams:
    team_id = team["id"]
    project_name = team.get("displayName", "Unknown Team")
    #print(f"\n🔍 Fetching channels for team: {project_name} ({team_id})")
    
    channels = get_team_channels(team_id, headers)
    
    for channel in channels:
        channel_name = channel["displayName"]
        channel_id = channel["id"]
        detected_planner_tabs = []
        
        tabs = get_channel_tabs(team_id, channel_id, headers)
        for tab in tabs:
            tab_name = tab.get("displayName", "")
            planner_id = tab.get("configuration", {}).get("entityId")
            planner_url = tab.get("webUrl")
            
            extracted_plan_id = extract_plan_id_from_planner_url(planner_url)
            if extracted_plan_id:
                planner_id = extracted_plan_id
            else:
                planner_id = clean_plan_id(planner_id)
            
            if planner_id and is_planner_tab(tab_name, planner_id, planner_url):
                detected_planner_tabs.append({
                    "tab_name": tab_name.encode("utf-8").decode("unicode_escape"),
                    "planner_id": planner_id,
                    "planner_url": planner_url,
                })
                #print(f"✅ Planner Tab Found: {tab_name} ({planner_id})")
        
        if detected_planner_tabs:
            channel_planners_detected.append({
                "project_name": project_name,
                "team_id": team_id,
                "channel_name": channel_name.encode("utf-8").decode("unicode_escape"),
                "channel_id": channel_id,
                "planner_tabs": detected_planner_tabs
            })
    
    time.sleep(1)  # Avoid hitting API rate limits

In [ ]:
# Split and save planner tabs
all_split_data = []
for item in channel_planners_detected:
    all_split_data.extend(split_planner_tabs(item))

# Convert back to JSON string for output
output_json = json.dumps(all_split_data, indent=4, ensure_ascii=False)

# #print or save the output
#print(output_json)

# Optional: Save the output to a new file
with open("teams_planner_data.json", "w", encoding="utf-8") as output_file:
    output_file.write(output_json)
